# Notebook 2 | Marginal Probability (Sum Rule)

In [1]:
from foundations_of_probability_and_statistics.cars.car_distribution import create_car_distribution
from foundations_of_probability_and_statistics.cars.car_distribution import create_joint_car_distribution

import pandas as pd

# show all rows of data frames and series per default
pd.set_option("display.max_rows", None)

## The sum rule: forgetting variables by summing

Given the joint of notebook 1, how do we recover the probability of a single variable, forgetting the rest? Marginalizing sums the joint $p(B = b, H = h, C = c)$ over the variables we no longer care about. For the brand marginal, in full form first and then in shorthand,

$$p(B = b) = \sum_h \sum_c p(B = b, H = h, C = c), \qquad p(b) = \sum_h \sum_c p(b, h, c).$$

The same rule gives any marginal, e.g. $p(C = c) = \sum_b \sum_h p(b, h, c)$. Reach for it whenever a question mentions fewer variables than the model tracks. It reappears as the denominator of Bayes' theorem (notebook 9) and as the leaf sums of probability trees (notebook 8).

## Derivation

1. Start from the joint $p(b, h, c) = p(b) \, p(h \mid b) \, p(c \mid b)$ of notebook 1.
2. Fix the value of interest, e.g. $B = \text{Porsche}$, and sum over all $(h, c)$ pairs.
3. Factor the brand term out: $\sum_h \sum_c p(b) \, p(h \mid b) \, p(c \mid b) = p(b) \sum_h p(h \mid b) \sum_c p(c \mid b)$, using conditional independence to split the double sum.
4. Both inner sums equal one because each conditional table is normalized, so $p(\text{Porsche}) = 0.3 \cdot 1 \cdot 1 = 0.3$.

The characteristic mistake is summing raw conditional entries across brands without the $p(b)$ weights; that answers a different question (an unweighted average over brands, not the population marginal).

## Worked example (by hand)

Factorized form, with the inner sums made explicit:

$$p(\text{Porsche}) = 0.3 \cdot 1 \cdot 1 = 0.3.$$

One full explicit state sum over the three red states (only Ferrari comes in red, so the other brands contribute zero):

$$p(\text{red}) = 0.2 \cdot 0.3 \cdot 0.6 + 0.2 \cdot 0.4 \cdot 0.6 + 0.2 \cdot 0.3 \cdot 0.6 = 0.036 + 0.048 + 0.036 = 0.12.$$

Comparing the two computations shows the two faces of the same rule: factorized when the structure helps, state by state when we want to see every contributor.

In [2]:
import math

joint = create_joint_car_distribution()

# factorized brand marginal: p(Porsche) = 0.3 * 1 * 1
brand_marginal = joint.groupby(level="brand").sum()
assert math.isclose(brand_marginal["Porsche"], 0.3 * 1 * 1)
assert math.isclose(brand_marginal["Porsche"], 0.3)

# explicit three-state sum over the red states
red = joint.xs("red", level="color")
assert len(red) == 3
assert math.isclose(red.loc[("Ferrari", 400)], 0.036)
assert math.isclose(red.loc[("Ferrari", 500)], 0.048)
assert math.isclose(red.loc[("Ferrari", 600)], 0.036)
assert math.isclose(red.sum(), 0.036 + 0.048 + 0.036)
assert math.isclose(red.sum(), 0.12)
brand_marginal

brand
Ferrari    0.2
Porsche    0.3
VW         0.5
Name: p, dtype: float64

## Generalization

Marginalizing over different variable subsets gives every one- and two-dimensional margin of the joint. All of them stay normalized, and marginalizing the joint over horsepower and color recovers the brand prior -- a consistency check worth running whenever a joint is built by hand. Notebook 3 conditions instead of summing: same slice of the joint, but renormalized rather than added up.

In [3]:
horsepower_marginal = joint.groupby(level="horsepower").sum()
color_marginal = joint.groupby(level="color").sum()

# spot checks against the factorized model
assert math.isclose(horsepower_marginal.loc[300], 0.5 * 0.1 + 0.3 * 0.4)
assert math.isclose(color_marginal["black"], 0.5 * 0.3 + 0.3 * 0.4 + 0.2 * 0.2)
assert math.isclose(horsepower_marginal.sum(), 1.0)
assert math.isclose(color_marginal.sum(), 1.0)
horsepower_marginal

horsepower
100    0.300
200    0.150
300    0.170
400    0.180
500    0.110
600    0.075
700    0.015
Name: p, dtype: float64

## References

- Blitzstein, J. K., Hwang, J. (2019): "Introduction to Probability", 2nd ed., Chapman & Hall/CRC, chapters "Random Variables and Their Distributions" and "Joint Distributions", https://www.routledge.com/Introduction-to-Probability-Second-Edition/Blitzstein-Hwang/p/book/9781138369917 (free PDF: https://probabilitybook.net/).
- Wasserman, L. (2004): "All of Statistics: A Concise Course in Statistical Inference", Springer Texts in Statistics, chapter "Probability", https://doi.org/10.1007/978-0-387-21736-9.
- scipy.stats.rv_discrete, https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rv_discrete.html.